In [ ]:
"""Predict dPdK on AMIP models using trained HadGEM ensemble and compare to leave-one-out AMIP mean."""

import glob
import json
import os
from pathlib import Path

import numpy as np
import torch
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from unet import ProbUNet

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
base_channels = 200
gn_groups = 1
kernel_size = 3
num_bins = 64
lat_dim = 128
batch_size = 11

dP_min = -700
dP_max = 1200

ens_name = (
    f"unet_ens_HG789_PR_dPdK_Softmax_unet6R_ch{base_channels}_k{kernel_size}_"
    f"{lat_dim}x_dPbins{num_bins}_gn{gn_groups}_dpmin{dP_min}_dPmax{dP_max}_sigma0.6"
)
ens_dir = Path("/Users/ewellmeyer/Documents/research/weights") / ens_name

amip_pr_path = "/Users/ewellmeyer/Documents/research/AMIP/AMIP_PR_his_rg128.nc"
amip_dlogpdk_dir = "/Users/ewellmeyer/Documents/research/AMIP/regridded/rg_128x192/dlogPdK"
landmask_file = "/Users/ewellmeyer/Documents/research/HadGEM/hadgem_landmask_rg128.nc"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================================================
# 2. LOAD MODEL METADATA
# ==============================================================================
with open(ens_dir / "norm_stats.json", "r") as f:
    ns = json.load(f)
x_mean = np.array(ns["x_mean"], dtype=np.float32)
x_std = np.array(ns["x_std"], dtype=np.float32)
y_mean = float(ns["y_mean"])
y_std = float(ns["y_std"])

with open(ens_dir / "born_bins.json", "r") as f:
    bin_info = json.load(f)
bin_centers = np.array(bin_info["bin_centers_norm"], dtype=np.float32)
bin_centers_t = torch.as_tensor(bin_centers, dtype=torch.float32, device=device).view(1, -1, 1, 1)

# ==============================================================================
# 3. LOAD AMIP DATA
# ==============================================================================
# Historic precipitation (mm/yr)
ds_pr = xr.open_dataset(amip_pr_path)
amip_pr = ds_pr['PR'].values  # (11, 128, 192)
model_names = [str(r) for r in ds_pr['realization'].values]
lats = ds_pr['latitude'].values
lons = ds_pr['longitude'].values
ds_pr.close()

# Short model names for display
short_names = [m.split('_')[0] for m in model_names]
n_models = len(model_names)
print(f"AMIP models ({n_models}): {short_names}")

# Load dlogPdK files and convert to dPdK = PR_his * dlogPdK
dlogpdk_files = sorted(glob.glob(os.path.join(amip_dlogpdk_dir, "*.nc")))
assert len(dlogpdk_files) == n_models, f"Expected {n_models} files, found {len(dlogpdk_files)}"

amip_dpdk = np.zeros((n_models, 128, 192), dtype=np.float32)
for i, (name, fpath) in enumerate(zip(model_names, dlogpdk_files)):
    ds = xr.open_dataset(fpath)
    dlogpdk = ds['dlogPdK'].values.squeeze()  # (128, 192)
    ds.close()
    # dPdK = P_his * dlogPdK  (since dlogPdK = (1/P)(dP/dK))
    amip_dpdk[i] = amip_pr[i] * dlogpdk
    print(f"  {short_names[i]}: dPdK range [{amip_dpdk[i].min():.1f}, {amip_dpdk[i].max():.1f}]")

# Landmask and lat weights
ds_lm = xr.open_dataset(landmask_file)
landmask = ds_lm['land_mask'].values.astype(bool)
ds_lm.close()

lat_weights = np.cos(np.deg2rad(lats)).astype(np.float32)
lat_weights = lat_weights / lat_weights.mean()

# ==============================================================================
# 4. NN PREDICTION ON AMIP INPUTS
# ==============================================================================
# Normalize inputs using HadGEM training stats
X_amip = amip_pr[:, np.newaxis, :, :]  # (11, 1, 128, 192)
X_amip_norm = (X_amip - x_mean[None, :, None, None]) / x_std[None, :, None, None]

# Load ensemble members
model_files = sorted(glob.glob(str(ens_dir / f"{ens_dir.name}_member*.pth")))
n_ens = len(model_files)
print(f"\nLoading {n_ens} ensemble members...")

models = []
for path in model_files:
    model = ProbUNet(1, base_channels, kernel_size, 0.0, num_bins, gn_groups=gn_groups).to(device)
    ckpt = torch.load(path, map_location=device)
    state = ckpt['model'] if 'model' in ckpt else ckpt
    model.load_state_dict(state, strict=False)
    model.eval()
    models.append(model)

# Run inference: store per-member predictions
all_mu = np.zeros((n_ens, n_models, 128, 192), dtype=np.float32)
xb = torch.as_tensor(X_amip_norm, dtype=torch.float32, device=device)

for m_idx, model in enumerate(models):
    with torch.inference_mode():
        probs = model.forward_components(xb).float()
        mu_norm = (probs * bin_centers_t).sum(dim=1)
        mu = mu_norm * y_std + y_mean
    all_mu[m_idx] = mu.cpu().numpy()

# Ensemble mean prediction
nn_pred = all_mu.mean(axis=0)  # (11, 128, 192)
print(f"NN prediction shape: {nn_pred.shape}")

In [ ]:
# ==============================================================================
# 5. LEAVE-ONE-OUT AMIP MEAN BASELINE
# ==============================================================================
# For each model i, the LOO baseline is the mean of all OTHER models' dPdK
amip_loo_mean = np.zeros_like(amip_dpdk)
for i in range(n_models):
    mask = np.ones(n_models, dtype=bool)
    mask[i] = False
    amip_loo_mean[i] = amip_dpdk[mask].mean(axis=0)

# ==============================================================================
# 6. COMPUTE RMSE: NN vs LOO baseline
# ==============================================================================
denom_land = float((landmask * lat_weights[:, None]).sum() + 1e-12)

def weighted_rmse(pred, truth, weights):
    """Per-sample weighted RMSE."""
    se = (pred - truth) ** 2 * weights[None, :, None]
    return np.sqrt(se.mean(axis=(1, 2)))

def weighted_rmse_land(pred, truth, weights, lmask):
    """Per-sample land-only weighted RMSE."""
    denom = float((lmask * weights[:, None]).sum() + 1e-12)
    se = (pred - truth) ** 2 * weights[None, :, None] * lmask[None]
    return np.sqrt(se.sum(axis=(1, 2)) / denom)

nn_rmse = weighted_rmse(nn_pred, amip_dpdk, lat_weights)
loo_rmse = weighted_rmse(amip_loo_mean, amip_dpdk, lat_weights)
nn_rmse_land = weighted_rmse_land(nn_pred, amip_dpdk, lat_weights, landmask)
loo_rmse_land = weighted_rmse_land(amip_loo_mean, amip_dpdk, lat_weights, landmask)

print(f"{'Model':<22s} {'NN RMSE':>10s} {'LOO RMSE':>10s} {'Improv%':>8s}  |  {'NN Land':>10s} {'LOO Land':>10s} {'Improv%':>8s}")
print("-" * 95)
for i in range(n_models):
    imp = (1 - nn_rmse[i] / (loo_rmse[i] + 1e-12)) * 100
    imp_l = (1 - nn_rmse_land[i] / (loo_rmse_land[i] + 1e-12)) * 100
    print(f"{short_names[i]:<22s} {nn_rmse[i]:10.2f} {loo_rmse[i]:10.2f} {imp:+8.1f}%  |  {nn_rmse_land[i]:10.2f} {loo_rmse_land[i]:10.2f} {imp_l:+8.1f}%")

print(f"\n{'MEAN':<22s} {nn_rmse.mean():10.2f} {loo_rmse.mean():10.2f} {(1-nn_rmse.mean()/(loo_rmse.mean()+1e-12))*100:+8.1f}%  |  {nn_rmse_land.mean():10.2f} {loo_rmse_land.mean():10.2f} {(1-nn_rmse_land.mean()/(loo_rmse_land.mean()+1e-12))*100:+8.1f}%")

In [ ]:
# ==============================================================================
# 7. SUMMARY FIGURES
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x_pos = np.arange(n_models)
width = 0.35

# Global RMSE bar chart
ax = axes[0]
ax.bar(x_pos - width/2, loo_rmse, width, label='LOO AMIP Mean', color='steelblue', alpha=0.8)
ax.bar(x_pos + width/2, nn_rmse, width, label='NN Prediction', color='darkorange', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(short_names, rotation=45, ha='right')
ax.set_ylabel('RMSE (mm yr$^{-1}$ K$^{-1}$)')
ax.set_title('Global RMSE')
ax.legend()

# Land RMSE bar chart
ax = axes[1]
ax.bar(x_pos - width/2, loo_rmse_land, width, label='LOO AMIP Mean', color='steelblue', alpha=0.8)
ax.bar(x_pos + width/2, nn_rmse_land, width, label='NN Prediction', color='darkorange', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(short_names, rotation=45, ha='right')
ax.set_ylabel('RMSE (mm yr$^{-1}$ K$^{-1}$)')
ax.set_title('Land-Only RMSE')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 8. MAP COMPARISON: per-model truth, NN pred, LOO mean, errors
# ==============================================================================
vmin, vmax = -300, 300
err_vmin, err_vmax = -200, 200
proj = ccrs.Robinson()

for i in range(n_models):
    fig, axes = plt.subplots(2, 2, figsize=(16, 8), subplot_kw={'projection': proj})
    fig.suptitle(f"{short_names[i]}", fontsize=14, fontweight='bold')

    panels = [
        (axes[0, 0], amip_dpdk[i], 'Truth (dPdK)', vmin, vmax, 'RdBu_r'),
        (axes[0, 1], nn_pred[i], 'NN Prediction', vmin, vmax, 'RdBu_r'),
        (axes[1, 0], amip_loo_mean[i], 'LOO AMIP Mean', vmin, vmax, 'RdBu_r'),
        (axes[1, 1], nn_pred[i] - amip_dpdk[i], 'NN Error', err_vmin, err_vmax, 'RdBu_r'),
    ]

    for ax, data, title, v0, v1, cmap in panels:
        ax.set_global()
        ax.coastlines(linewidth=0.5)
        im = ax.pcolormesh(
            lons, lats, data,
            transform=ccrs.PlateCarree(),
            cmap=cmap, vmin=v0, vmax=v1,
        )
        ax.set_title(title)
        plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)

    plt.tight_layout()
    plt.show()